Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  #  local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,

)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.15 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
from google.colab import files
files.upload()

Saving ML-cleaned_file.jsonl to ML-cleaned_file.jsonl


{'ML-cleaned_file.jsonl': b'{"instruction": "Extract entities using ONLY the following labels: TOPIC, CONCEPT, METHOD, ALGORITHM, OTHER.", "input": "Data preprocessing is a crucial data mining and machine learning process that converts raw, messy, or incomplete data into a clean, structured format for analysis. Key steps include cleaning (handling missing values, outliers, duplicates), data transformation (normalization, encoding), and reduction, which enhance model accuracy and efficiency.", "output": "[{\\"entity\\": \\"Data preprocessing\\", \\"label\\": \\"TOPIC\\"}, {\\"entity\\": \\"machine learning\\", \\"label\\": \\"TOPIC\\"}, {\\"entity\\": \\"accuracy\\", \\"label\\": \\"CONCEPT\\"}, {\\"entity\\": \\"outliers\\", \\"label\\": \\"CONCEPT\\"}, {\\"entity\\": \\"normalization\\", \\"label\\": \\"METHOD\\"}]"}\n{"instruction": "Extract entities using ONLY the following labels: TOPIC, CONCEPT, METHOD, ALGORITHM, OTHER.", "input": "Data cleaning is the process of fixing or removi

Data Prep

In [5]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
# dataset = load_dataset("unsloth/alpaca-cleaned", split = "train")
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="ML-cleaned_file.jsonl",
    split="train"
)

# Split into train & test
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset  = dataset["test"]

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1186 [00:00<?, ? examples/s]

Train the model

In [6]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1186 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [7]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
6.707 GB of memory reserved.


In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,186 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.177800
2,2.093600
3,2.373700
4,2.087600
5,1.834600
6,1.404700
7,1.354800
8,1.246400
9,1.148800
10,0.900100


Evaluation

In [9]:
import json
from tqdm import tqdm
from collections import defaultdict

model.eval()

predictions = []
references = []

# ---------------- Generate Predictions ----------------
for example in tqdm(test_dataset):

    # Prepare prompt without output
    prompt = alpaca_prompt.format(
        example["instruction"],
        example["input"],
        ""
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract model's response only
    prediction = decoded.split("### Response:")[-1].strip()

    predictions.append(prediction)
    references.append(example["output"])


# ---------------- Safe JSON Loader ----------------
def safe_json_load(x):
    try:
        if isinstance(x, str):
            return json.loads(x)
        elif isinstance(x, list):
            return x
        else:
            return []
    except:
        return []

pred_entities = [safe_json_load(p) for p in predictions]
true_entities = [safe_json_load(t) for t in references]


# ---------------- Compute Metrics ----------------
def compute_metrics(preds, refs, partial_threshold=0.5):
    tp = 0
    fp = 0
    fn = 0
    exact_matches = 0
    partial_matches = 0

    # Per-label stats
    label_tp = defaultdict(int)
    label_fp = defaultdict(int)
    label_fn = defaultdict(int)

    for pred, ref in zip(preds, refs):
        pred_set = {(e["entity"], e["label"]) for e in pred}
        ref_set  = {(e["entity"], e["label"]) for e in ref}

        # Overall stats
        tp += len(pred_set & ref_set)
        fp += len(pred_set - ref_set)
        fn += len(ref_set - pred_set)

        # Exact match
        if pred_set == ref_set:
            exact_matches += 1

        # Partial match
        if len(ref_set) == 0:
            continue
        match_ratio = len(pred_set & ref_set) / len(ref_set)
        if match_ratio >= partial_threshold:
            partial_matches += 1

        # Per-label stats
        labels = set([e["label"] for e in pred] + [e["label"] for e in ref])
        for label in labels:
            pred_label = {e for e in pred_set if e[1]==label}
            ref_label  = {e for e in ref_set if e[1]==label}
            label_tp[label] += len(pred_label & ref_label)
            label_fp[label] += len(pred_label - ref_label)
            label_fn[label] += len(ref_label - pred_label)

    # Overall metrics
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0
    micro_f1  = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    exact_acc = exact_matches / len(refs)
    partial_acc = partial_matches / len(refs)

    # Per-label F1
    per_label_metrics = {}
    for label in label_tp.keys():
        p = label_tp[label] / (label_tp[label] + label_fp[label]) if (label_tp[label] + label_fp[label])>0 else 0
        r = label_tp[label] / (label_tp[label] + label_fn[label]) if (label_tp[label] + label_fn[label])>0 else 0
        f = 2*p*r/(p+r) if p+r>0 else 0
        per_label_metrics[label] = {"precision": p, "recall": r, "f1": f}

    # Macro-F1 = average of per-label F1
    macro_f1 = sum(m["f1"] for m in per_label_metrics.values()) / len(per_label_metrics) if per_label_metrics else 0

    return precision, recall, micro_f1, exact_acc, partial_acc, macro_f1, per_label_metrics


# ---------------- Run Metrics ----------------
precision, recall, micro_f1, exact_acc, partial_acc, macro_f1, per_label_metrics = compute_metrics(pred_entities, true_entities)

# ---------------- Print Results ----------------
print("=== Overall Metrics ===")
print(f"Micro Precision: {precision:.4f}")
print(f"Micro Recall:    {recall:.4f}")
print(f"Micro F1:        {micro_f1:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
# print(f"Exact Match Accuracy: {exact_acc:.4f}")
print(f"Partial Match Accuracy (>50% entities correct): {partial_acc:.4f}")

print("\n=== Per-Label Metrics ===")
for label, metrics in per_label_metrics.items():
    print(f"{label}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1']:.4f}")

100%|██████████| 297/297 [20:19<00:00,  4.11s/it]

=== Overall Metrics ===
Micro Precision: 0.5904
Micro Recall:    0.5476
Micro F1:        0.5682
Macro F1:        0.4498
Partial Match Accuracy (>50% entities correct): 0.6229

=== Per-Label Metrics ===
TOPIC: Precision=0.6148, Recall=0.6748, F1=0.6434
CONCEPT: Precision=0.6288, Recall=0.5896, F1=0.6086
METHOD: Precision=0.4000, Recall=0.2687, F1=0.3214
ALGORITHM: Precision=0.3636, Recall=0.3077, F1=0.3333
OTHER: Precision=0.4043, Recall=0.2969, F1=0.3423


In [10]:
print(f"Exact Match Accuacy :{exact_acc:.4f}")

Exact Match Accuacy :0.0976


Saving, loading finetuned models

In [ ]:
# --------------------- SAVE LOADED / FINETUNED MODEL ---------------------
# This will save your trained LoRA adapters and optionally merge into full model for inference

from unsloth import FastLanguageModel

# Path to save LoRA adapters
lora_save_path = "llama_lora"  # just the adapters
# Path to save merged model (optional)
merged_16bit_path = "llama_finetune_16bit"
merged_4bit_path  = "llama_finetune_4bit"

# --------------------- 1️⃣ Save LoRA adapters only ---------------------
model.save_pretrained(lora_save_path)
tokenizer.save_pretrained(lora_save_path)
print(f"LoRA adapters saved locally at '{lora_save_path}'")

# --------------------- 2️⃣ Merge LoRA into base model (16-bit) ---------------------
model.save_pretrained_merged(
    merged_16bit_path,
    tokenizer,
    save_method="merged_16bit"
)
print(f"Merged 16-bit model saved locally at '{merged_16bit_path}'")

# --------------------- 3️⃣ Merge LoRA into base model (4-bit, optional) ---------------------
model.save_pretrained_merged(
    merged_4bit_path,
    tokenizer,
    save_method="merged_4bit_forced"
)
print(f"Merged 4-bit model saved locally at '{merged_4bit_path}'")

LoRA adapters saved locally at 'llama_lora'


config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

In [ ]:
!rm -rf /content/outputs